# M3 decision reports on Kaggle

We run `serpentguard.decision.decision_report` on every training run whose kernel output is attached, to turn each
run's outputs into the three verdicts. For each run it fits the verdict thresholds on the calibration outputs,
reports the verdicts on the validation and unknown-pool outputs, the latter for unknown-species detection (U1), and
writes `decision_metrics.json`.

**Inputs:** the dataset `serpentguard-indian-snakes-clean`, found with `kagglehub.dataset_download`, and the outputs
of the kernels in `RUN_KERNELS`, read through `kagglehub.notebook_output_download`. A kernel can read only its own
account's private outputs, so each account runs its own copy; every kernel handle also takes `"dataset:owner/slug"`,
a private dataset copy of another account's output. `QUALITY_KERNEL` (notebook 17's `quality.csv`, for the poor-image
reason) and `PREDS_KERNEL` (notebook 13's `frontend_preds.csv`) are optional.

**Outputs** in `/kaggle/working/decision/`: one folder per run with its `decision_metrics.json`, and
`decision_summary.csv` with the main verdict numbers of every run.

**Run:** on Kaggle, CPU only. It runs there rather than locally so the saved outputs (`*.npz`) never have to be
downloaded.

- `SCORER` picks what ranks the species, the species head (`"head"`) or the nearest prototype (`"prototype"`), for
  the fit and every verdict; the notebook can be run once with each.
- With `PREDS_KERNEL`, a run that reads predicted masks or boxes (`mask_source: predicted`, F1, F1m, F2, F3) gets
  `tau_snake` from the front end's box confidences, so its report counts the photos with no snake found.
- `RECALL_BOUND` is `"point"` to fit `theta_d` and `theta_who` to 95% WHO recall on the calibration set, or
  `"lower"` to fit them so the photographer-bootstrap lower bound of that recall reaches 95%; validation never enters
  the fit.
- `DR_RUN` (optional, a run id such as `"H1_top1_s1"`) also runs the decision-rule study DR on that run: which danger
  rule calls the fewest snakes outside the WHO list and the viper and elapid families dangerous at the same WHO
  recall. It writes `dr/<run id>/danger_rules.json` and `dr/m3_decision_rules_<run id>.png`.

In [1]:
RUN_KERNELS = ['dataset:puneetbajajiitm/serpentguard-m3-run-final']
QUALITY_KERNEL = 'puneetbajajiitm/sg-m3-r1final'
PREDS_KERNEL = 'puneetbajajiitm/sg-m3-frontends'
SCORER = 'head'
RECALL_BOUND = 'lower'
DR_RUN = 'H1_top1_s1'

In [2]:
import subprocess
import sys

SCIKIT_LEARN_VERSION = "1.9.1"  # requirements-m3.txt; every metric uses it
subprocess.run([sys.executable, "-m", "pip", "install", "-q", f"scikit-learn=={SCIKIT_LEARN_VERSION}"], check=True)

import sklearn

assert sklearn.__version__ == SCIKIT_LEARN_VERSION, f"scikit-learn {sklearn.__version__} is loaded"

In [3]:
import json
import shutil
from pathlib import Path

import kagglehub
import pandas as pd

OWNER = "puneetbajajiitm"
CODE = Path("/kaggle/input/serpentguard-code/src")
sys.path.insert(0, str(CODE if CODE.exists() else Path("../src").resolve()))

from serpentguard.config import load_config
from serpentguard.decision import decision_report
from serpentguard.run_data import FRONTEND_PREDS_FILE, PREDICTED

RUN_FILES = ("val_outputs.npz", "calib_outputs.npz", "unknown_outputs.npz", "prototypes.npz", "final_metrics.json",
             "config.yaml")
OUT = Path("/kaggle/working/decision")

data_root = Path(kagglehub.dataset_download(f"{OWNER}/serpentguard-indian-snakes-clean"))
classes = pd.read_csv(data_root / "classes.csv")
metadata = pd.read_csv(data_root / "metadata.csv", usecols=["path", "split"])


DATASET_PREFIX = "dataset:"  # "dataset:owner/slug": a private dataset copy of a kernel's output, for another account


def kernel_output(handle):
    """A kernel's output folder, or for "dataset:owner/slug" a private dataset copy of it."""
    if handle.startswith(DATASET_PREFIX):
        return Path(kagglehub.dataset_download(handle.removeprefix(DATASET_PREFIX)))
    return Path(kagglehub.notebook_output_download(handle))


def run_folders(root):
    return sorted(p.parent for p in root.rglob("val_outputs.npz") if (p.parent / "calib_outputs.npz").exists())


quality = None
if QUALITY_KERNEL:
    found = sorted(kernel_output(QUALITY_KERNEL).rglob("quality.csv"))
    quality = pd.read_csv(found[0]) if found else None
    print("quality table:", found[0] if found else "not found")

box_conf = None
if PREDS_KERNEL:
    preds_files = sorted(kernel_output(PREDS_KERNEL).rglob(FRONTEND_PREDS_FILE))
    assert len(preds_files) == 1, f"{PREDS_KERNEL}'s output must hold one {FRONTEND_PREDS_FILE}; found {preds_files}"
    preds = pd.read_csv(preds_files[0], usecols=["path", "box_conf"])
    box_conf = dict(zip(preds["path"], preds["box_conf"]))
    print(f"front-end predictions: {preds_files[0]}, {preds['box_conf'].notna().sum():,} photos with a box")

quality table: /kaggle/input/notebooks/puneetbajajiitm/sg-m3-r1final/robustness/quality.csv
front-end predictions: /kaggle/input/notebooks/puneetbajajiitm/sg-m3-frontends/frontend_preds/frontend_preds.csv, 39,514 photos with a box


In [4]:
rows = []
for handle in RUN_KERNELS:
    for source in run_folders(kernel_output(handle)):
        run = OUT / source.name
        run.mkdir(parents=True, exist_ok=True)
        for name in RUN_FILES:
            if (source / name).exists():
                shutil.copy2(source / name, run / name)
        predicted = load_config(run / "config.yaml").mask_source == PREDICTED
        if predicted and box_conf is None:
            print(f"{source.name} reads predicted masks or boxes; without PREDS_KERNEL it gets no tau_snake")
        report = decision_report(run, classes, metadata=metadata, quality=quality, scorer=SCORER,
                                 box_conf=box_conf if predicted else None, recall_bound=RECALL_BOUND)
        for name in RUN_FILES:
            if name.endswith(".npz"):
                (run / name).unlink()
        metrics, fit = report["verdict_metrics"], report["fit"]
        rows.append({"run": source.name, "kernel": handle, "scorer": report["scorer"],
                     "recall_bound": report["recall_bound"],
                     "recall_target": report.get("recall_bound_fit", {}).get("target"),
                     "who_dangerous_recall": metrics["who_dangerous_recall"], "false_alarm": metrics["false_alarm"],
                     "venomous_recall": metrics["venomous_recall"],
                     "o3_all": metrics["o3"]["dangerous_as_not_listed_all"], "abstain_rate": metrics["abstain_rate"],
                     "unknown_flagged": report["u1"]["groups"]["all"]["flagged_at_tau"],
                     "unknown_auroc_msp": report["u1"]["groups"]["all"]["open_set"]["msp"]["auroc"],
                     "theta_d": report["thresholds"]["theta_d"], "theta_who": report["thresholds"]["theta_who"],
                     "calibration_recall": fit["who_dangerous_recall"],
                     "tau_snake": report["no_snake"].get("tau_snake"), "no_snake_val": report["no_snake"]["val"]})
        print(f"{source.name}: WHO recall {metrics['who_dangerous_recall']:.3f}, false alarms "
              f"{metrics['false_alarm']:.3f}, unknown flagged {rows[-1]['unknown_flagged']:.3f}")
summary = pd.DataFrame(rows)
summary.to_csv(OUT / "decision_summary.csv", index=False)
assert len(summary), "no run folders found in the attached kernel outputs"
summary

H1_top1_s1: WHO recall 0.948, false alarms 0.311, unknown flagged 0.075


,run,kernel,scorer,recall_bound,recall_target,who_dangerous_recall,false_alarm,venomous_recall,o3_all,abstain_rate,unknown_flagged,unknown_auroc_msp,theta_d,theta_who,calibration_recall,tau_snake,no_snake_val
0,H1_top1_s1,dataset:puneetbajajiitm/serpentguard-m3-run-final,head,lower,0.965,0.947912,0.310731,0.944603,0.020648,0.095388,0.075388,0.692276,0.056606,inf,0.966431,0.102,151


## Decision-rule study (DR)

Only with `DR_RUN`. Every rule (`RULE_TEXT`) is fitted on the run's calibration outputs at each WHO recall target in
`TARGETS`, both at the target itself and at its photographer-bootstrap lower bound, and measured once on validation.
A rule wins (`WIN_RULE`) only if its validation false alarms at 0.95 sit below DR0's with non-overlapping intervals.

In [5]:
# DR study helpers (1/2): the rules' scores, their fits on calibration and their shares on validation
import math
from dataclasses import replace

import numpy as np
import pandas as pd
from scipy.special import entr, logit
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import StratifiedGroupKFold
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler

from serpentguard.decision import (DANGEROUS, MEDICAL_CSV, NO_SNAKE_NOT_AVAILABLE, NOT_LISTED, NOT_WHO, PLAIN,
                                   RECALL_BOUNDS, SNAKE_THRESHOLD_SOURCE, UNKNOWN, WHO_RECALL_TARGET,
                                   _box_conf_values, _calibrated, _check_finite, _decide, _joint_fit, _load,
                                   danger_fit, danger_tables, fit_snake_threshold, listed_and_outside, photographers,
                                   signals)
from serpentguard.metrics import CI_PERCENTILES, _share, grouped_resamples
from serpentguard.run_files import CALIBRATION_OUTPUTS, PROTOTYPES_FILE, VAL_OUTPUTS
from serpentguard.run_outputs import write_json

DR0, DR1, DR2_K3, DR2_K5, DR3_MAX, DR3_MIX, DR4 = "DR0", "DR1", "DR2_top3", "DR2_top5", "DR3_max", "DR3_mix", "DR4"
RULES = (DR0, DR1, DR2_K3, DR2_K5, DR3_MAX, DR3_MIX, DR4)
RULE_TEXT = {
    DR0: "the current rule: listed mass >= theta_d or WHO head >= theta_who, fitted jointly",
    DR1: "logistic regression on the outputs (FEATURES) >= theta_d, cross-fitted by photographer; replaces both "
         "signals",
    DR2_K3: "listed mass among the top 3 species >= theta_d, jointly with the WHO head",
    DR2_K5: "listed mass among the top 5 species >= theta_d, jointly with the WHO head",
    DR3_MAX: "max(species listed mass, genus head's listed mass) >= theta_d, jointly with the WHO head",
    DR3_MIX: "weight x species listed mass + (1 - weight) x genus head's listed mass >= theta_d, jointly with the WHO "
             "head; weight fitted on calibration at each target's point fit",
    DR4: "DR0's thresholds; a call only a WHO signal makes, with listed mass and WHO head both under CONFIDENT_DANGER, "
         "becomes we don't know (keep away); a variant, since it changes what the verdicts mean",
}
TARGETS = (0.9, 0.95, 0.975)
TOP_K = {DR2_K3: 3, DR2_K5: 5}
FEATURES = ("listed_mass", "who_head", "top1_prob", "top1_listed", "entropy", "margin", "genus_listed_mass",
            "venomous_family")
LOGIT_FEATURES = ("listed_mass", "who_head", "top1_prob", "genus_listed_mass", "venomous_family")
PROBABILITY_CLIP = 1e-6       # probabilities are clipped to [clip, 1 - clip] before their logit
CROSS_FIT_FOLDS, CROSS_FIT_SEED = 5, 0
MIX_WEIGHTS = tuple(round(float(weight), 1) for weight in np.linspace(0, 1, 11))
CONFIDENT_DANGER = 0.5        # DR4: a WHO signal at or above this is a confident dangerous call
O3_LIMIT = 0.02


def listed_top_k_mass(probs, listed, k):
    """The probability of the WHO-listed species among each photo's k most probable species."""
    top = np.argsort(-probs, axis=1, kind="stable")[:, :k]
    return (np.take_along_axis(probs, top, axis=1) * listed[top]).sum(axis=1)


def genus_listed_mass(found, tables):
    """The genus head's probability on genera holding a WHO-listed species."""
    return found.genus_head[:, tables.listed_genus].sum(axis=1)


def with_score(found, score, head=True):
    """found with `score` where the rules read the listed mass, and without the WHO head unless `head`."""
    return replace(found, listed_mass=np.asarray(score, dtype=float), who_head=found.who_head if head else None)


def danger_features(found, tables):
    """DR1's features, probabilities as logits; a row whose species probabilities are not finite stays NaN."""
    probs = found.probs
    ranked = -np.sort(-probs, axis=1)
    columns = {"listed_mass": found.listed_mass, "who_head": found.who_listed, "top1_prob": ranked[:, 0],
               "top1_listed": (tables.who != NOT_WHO)[probs.argmax(axis=1)].astype(float),
               "entropy": entr(probs).sum(axis=1), "margin": ranked[:, 0] - ranked[:, 1],
               "genus_listed_mass": genus_listed_mass(found, tables),
               "venomous_family": found.family_head[:, tables.venomous_family].sum(axis=1)}
    values = np.column_stack([logit(np.clip(columns[name], PROBABILITY_CLIP, 1 - PROBABILITY_CLIP))
                              if name in LOGIT_FEATURES else columns[name] for name in FEATURES])
    values[~np.isfinite(probs).all(axis=1)] = math.nan
    return values


def learned_scores(found, val_found, tables, calib_outputs):
    """DR1: P(WHO-listed) from danger_features, trained on WHO-listed (1) against neither WHO-listed nor viper or elapid
    (0). Photographer-grouped K-fold: each calibration photo is scored by a model that never saw its photographer, and
    validation by the mean of the fold models. Returns (calibration, validation, mean standardised coefficients)."""
    listed, outside = listed_and_outside(np.asarray(calib_outputs["labels"], dtype=int), tables)
    y, train = listed.astype(int), listed | outside
    values, val_values = danger_features(found, tables), danger_features(val_found, tables)
    finite = np.isfinite(values).all(axis=1)
    out_of_fold, models = np.full(len(y), math.nan), []
    splitter = StratifiedGroupKFold(n_splits=CROSS_FIT_FOLDS, shuffle=True, random_state=CROSS_FIT_SEED)
    for fit_rows, held_rows in splitter.split(np.zeros(len(y)), y, photographers(calib_outputs)):
        fitting = np.intersect1d(fit_rows, np.flatnonzero(train & finite))
        model = make_pipeline(StandardScaler(), LogisticRegression(max_iter=1000)).fit(values[fitting], y[fitting])
        scored = held_rows[finite[held_rows]]
        out_of_fold[scored] = model.predict_proba(values[scored])[:, 1]
        models.append(model)
    val_finite = np.isfinite(val_values).all(axis=1)
    validation = np.full(len(val_values), math.nan)
    if val_finite.any():
        validation[val_finite] = np.mean([model.predict_proba(val_values[val_finite])[:, 1] for model in models], axis=0)
    coefficients = np.mean([model[-1].coef_[0] for model in models], axis=0).tolist()
    return out_of_fold, validation, dict(zip(FEATURES, coefficients))


def mix_weight(found, species, genus, base, tables, labels, target):
    """DR3_MIX's weight: the fewest calibration false alarms at its point fit, ties to the larger weight."""
    best = None
    for weight in MIX_WEIGHTS:
        fit = danger_fit(with_score(found, weight * species + (1 - weight) * genus), base, tables, labels, target)
        if fit is not None and (best is None or fit.false_alarm <= best[0]):
            best = (fit.false_alarm, weight)
    if best is None:
        raise ValueError(f"no mix weight reaches a WHO recall of {target} on calibration")
    return best[1]


def uncertain_to_unknown(found, thresholds, tables, snake_found=None):
    """DR4: a dangerous call that fails with both WHO thresholds raised to CONFIDENT_DANGER becomes we don't know."""
    low = _decide(found, thresholds, tables, snake_found).label
    raised = replace(thresholds, theta_d=max(thresholds.theta_d, CONFIDENT_DANGER),
                     theta_who=max(thresholds.theta_who, CONFIDENT_DANGER))
    high = _decide(found, raised, tables, snake_found).label
    return np.where((low == DANGEROUS) & (high != DANGEROUS), UNKNOWN, low)


def share_masks(labels, verdicts, tables):
    """Per reported share, the photos it is a share of and those it counts."""
    labels = np.asarray(labels, dtype=int)
    listed, outside = listed_and_outside(labels, tables)
    dangerous = verdicts == DANGEROUS
    return {"who_recall": (listed, dangerous), "false_alarm": (outside, dangerous),
            "false_alarm_outside_table": (~tables.dangerous[labels], dangerous),
            "o3": (listed, verdicts == NOT_LISTED), "unknown_share": (np.ones(len(labels), dtype=bool), verdicts == UNKNOWN)}


def verdict_shares(labels, verdicts, tables):
    return {name: _share(hit[among]) for name, (among, hit) in share_masks(labels, verdicts, tables).items()}


def verdict_intervals(verdicts_by_key, val_outputs, tables):
    """95% intervals of every share, all rules on the same photographer-bootstrap resamples."""
    groups, n = photographers(val_outputs), len(val_outputs["labels"])
    counts = np.array([np.bincount(rows, minlength=n) for rows in grouped_resamples(groups)], dtype=np.float32)
    intervals = {}
    with np.errstate(invalid="ignore", divide="ignore"):
        for key, verdicts in verdicts_by_key.items():
            intervals[key] = {}
            for name, (among, hit) in share_masks(val_outputs["labels"], verdicts, tables).items():
                shares = (counts @ (among & hit).astype(np.float32)).astype(float) / (counts @ among.astype(np.float32))
                intervals[key][name] = [float(value) for value in np.percentile(shares, CI_PERCENTILES)]
    return intervals


def rule_result(rule_signals, base, tables, calib, val, target, bound, snake):
    """One rule's calibration fit and validation shares at one target and bound, or why it can't be fitted."""
    calib_signals, val_signals = rule_signals
    try:
        fit, bounded = _joint_fit(calib_signals, base, tables, calib, target, bound)
    except ValueError as error:
        return {"unreachable": str(error)}, None
    labels = _decide(val_signals, fit.thresholds, tables, snake).label
    calibration = {"thresholds": {"theta_d": fit.thresholds.theta_d, "theta_who": fit.thresholds.theta_who},
                   "target": target if bounded is None else bounded["target"],
                   "lower_bound": None if bounded is None else bounded["lower_bound"],
                   "who_recall": fit.who_recall, "false_alarm": fit.false_alarm}
    return {"calibration": calibration, "validation": verdict_shares(val["labels"], labels, tables)}, labels


def dr4_result(found, val_found, tables, calib, val, d0_result, base, snake):
    """DR4 at DR0's thresholds for the same target and bound."""
    if "unreachable" in d0_result:
        return {"unreachable": "DR0 could not be fitted at this target"}, None
    thresholds = replace(base, **d0_result["calibration"]["thresholds"])
    calib_labels = uncertain_to_unknown(found, thresholds, tables)
    labels = uncertain_to_unknown(val_found, thresholds, tables, snake_found=snake)
    calibration = {**d0_result["calibration"], "confident": CONFIDENT_DANGER,
                   **{name: value for name, value in verdict_shares(calib["labels"], calib_labels, tables).items()
                      if name in ("who_recall", "false_alarm", "o3")}}
    return {"calibration": calibration, "validation": verdict_shares(val["labels"], labels, tables)}, labels

In [6]:
# DR study helpers (2/2): every rule at every target and bound, the summary and the figure
from matplotlib.figure import Figure

FIGURE_FILE = "m3_decision_rules_{}.png"
WIN_RULE = ("a rule wins when its validation false alarms at the 0.95 target sit below DR0's with non-overlapping 95% "
            "photographer-bootstrap intervals and its O3 is no worse; DR4 also needs its validation WHO recall (O2) to "
            "reach the target")
SURFACE, INK, MUTED, GRID = "#fcfcfb", "#0b0b0b", "#52514e", "#e4e3df"
RULE_COLOR = {DR0: INK, DR1: "#2a78d6", DR2_K3: "#eb6834", DR2_K5: "#1baf7a", DR3_MAX: "#eda100", DR3_MIX: "#e87ba4",
              DR4: "#6250d6"}
MARKER_SIZE, HEADLINE_MARKER_SIZE = 5.0, 8.5


def danger_rules_report(run_dir, classes, out, figure, box_conf=None):
    """Every rule at every target and recall bound, fitted on calibration and measured on validation (with box_conf's
    no_snake flags, which the fits never read); writes danger_rules.json into `out` and the figure."""
    tables = danger_tables(classes, pd.read_csv(MEDICAL_CSV, keep_default_na=False))
    calib, val = _load(run_dir / CALIBRATION_OUTPUTS), _load(run_dir / VAL_OUTPUTS)
    prototypes = np.load(run_dir / PROTOTYPES_FILE)[PLAIN]
    _check_finite({"calibration": calib, "val": val}, PLAIN, prototypes)
    found, base = _calibrated(calib, prototypes, tables)
    val_found = signals(val, base.temperature, base.temperature_who, prototypes, tables)
    snake, no_snake = None, {"source": NO_SNAKE_NOT_AVAILABLE}
    if box_conf is not None:  # the validation no_snake flags, as decision_report sets them
        tau_snake = fit_snake_threshold(_box_conf_values(box_conf, calib["paths"]))
        snake = _box_conf_values(box_conf, val["paths"]) >= tau_snake
        no_snake = {"source": SNAKE_THRESHOLD_SOURCE, "tau_snake": tau_snake, "val": int((~snake).sum())}
    learned, learned_val, coefficients = learned_scores(found, val_found, tables, calib)
    listed, pair = tables.who != NOT_WHO, (found, val_found)
    genus = tuple(genus_listed_mass(signal, tables) for signal in pair)
    scores = {DR0: pair, DR1: (with_score(found, learned, head=False), with_score(val_found, learned_val, head=False)),
              **{rule: tuple(with_score(signal, listed_top_k_mass(signal.probs, listed, k)) for signal in pair)
                 for rule, k in TOP_K.items()},
              DR3_MAX: tuple(with_score(signal, np.fmax(signal.listed_mass, mass)) for signal, mass in zip(pair, genus))}
    results, verdicts, weights = {rule: {} for rule in RULES}, {}, {}
    for target in TARGETS:
        weight = weights[str(target)] = mix_weight(found, found.listed_mass, genus[0], base, tables, calib["labels"],
                                                   target)
        scores[DR3_MIX] = tuple(with_score(signal, weight * signal.listed_mass + (1 - weight) * mass)
                                for signal, mass in zip(pair, genus))
        for bound in RECALL_BOUNDS:
            for rule in RULES:
                if rule == DR4:
                    result, labels = dr4_result(found, val_found, tables, calib, val, results[DR0][str(target)][bound],
                                                base, snake)
                else:
                    result, labels = rule_result(scores[rule], base, tables, calib, val, target, bound, snake)
                if rule == DR3_MIX and "calibration" in result:
                    result["calibration"]["weight"] = weight
                results[rule].setdefault(str(target), {})[bound] = result
                if labels is not None:
                    verdicts[rule, str(target), bound] = labels
    for (rule, target, bound), interval in verdict_intervals(verdicts, val, tables).items():
        results[rule][target][bound]["validation"]["intervals"] = interval
    out.mkdir(parents=True, exist_ok=True)
    report = write_json(out / "danger_rules.json", {**summary(run_dir.name, base, calib, val, coefficients, weights,
                                                              results), "no_snake": no_snake})
    plot_rules(report, figure)
    return report


def rule_wins(rule, result, reference, target):
    """WIN_RULE for one rule's result against DR0's at the same target and bound."""
    if "unreachable" in result or "unreachable" in reference:
        return False
    mine, theirs = result["validation"], reference["validation"]
    below = (mine["false_alarm"] < theirs["false_alarm"]
             and mine["intervals"]["false_alarm"][1] < theirs["intervals"]["false_alarm"][0])
    return below and mine["o3"] <= theirs["o3"] and (rule != DR4 or mine["who_recall"] >= target)


def headline_row(result):
    if "unreachable" in result:
        return result
    shown = result["validation"]
    return {"who_recall": shown["who_recall"], "false_alarm": shown["false_alarm"],
            "false_alarm_interval": shown["intervals"]["false_alarm"], "o3": shown["o3"],
            "o3_within_limit": shown["o3"] <= O3_LIMIT, "unknown_share": shown["unknown_share"],
            "false_alarm_outside_table": shown["false_alarm_outside_table"]}


def summary(run, base, calib, val, coefficients, weights, results):
    key = str(WHO_RECALL_TARGET)
    wins = {bound: [rule for rule in RULES[1:]
                    if rule_wins(rule, results[rule][key][bound], results[DR0][key][bound], WHO_RECALL_TARGET)]
            for bound in RECALL_BOUNDS}
    winners = [rule for rule in RULES[1:] if all(rule in wins[bound] for bound in RECALL_BOUNDS)]
    return {"run": run, "scorer": "head", "prototype_set": PLAIN,
            "photos": {"calibration": len(calib["labels"]), "val": len(val["labels"])},
            "temperature": base.temperature, "temperature_who": base.temperature_who, "rules": RULE_TEXT,
            "threshold_note": "theta_d is the threshold on each rule's own score, which stands in for the listed mass",
            "targets": list(TARGETS),
            "dr1": {"features": list(FEATURES), "folds": CROSS_FIT_FOLDS, "seed": CROSS_FIT_SEED,
                    "trained_on": "WHO-listed (1) against neither WHO-listed nor viper or elapid (0); every photo "
                                  "scored out of fold",
                    "coefficients": coefficients},
            "dr3_mix_weight": weights, "dr4_confident": CONFIDENT_DANGER, "o3_limit": O3_LIMIT,
            "results": results,
            "headline": {"target": WHO_RECALL_TARGET,
                         "rows": {rule: {bound: headline_row(results[rule][key][bound]) for bound in RECALL_BOUNDS}
                                  for rule in RULES}},
            "win_rule": WIN_RULE, "wins": wins, "winner": winners,
            "verdict": f"{', '.join(winners)} beat DR0 at both recall bounds" if winners else "no rule beats DR0"}


def plot_rules(report, path):
    """Validation false alarms against WHO recall, one line per rule through its fits at every target and bound."""
    figure = Figure(figsize=(8.0, 5.6), facecolor=SURFACE)
    axis = figure.subplots()
    axis.set_facecolor(SURFACE)
    ends = {}
    for rule in RULES:
        points = sorted((result["validation"]["who_recall"], result["validation"]["false_alarm"], target, bound)
                        for target, by_bound in report["results"][rule].items()
                        for bound, result in by_bound.items() if "unreachable" not in result)
        if not points:
            continue
        recall, alarm = [point[0] for point in points], [point[1] for point in points]
        axis.plot(recall, alarm, color=RULE_COLOR[rule], linewidth=2.0, label=rule, zorder=2)
        for x, y, target, bound in points:
            size = HEADLINE_MARKER_SIZE if float(target) == WHO_RECALL_TARGET else MARKER_SIZE
            axis.plot(x, y, marker="o", markersize=size, color=RULE_COLOR[rule], zorder=3,
                      markerfacecolor=RULE_COLOR[rule] if bound == "point" else SURFACE, markeredgewidth=1.5)
        ends[rule] = (recall[-1], alarm[-1])
    low, high = axis.get_ylim()  # direct labels right of each line's end, pushed up just enough not to overlap
    placed = None
    for rule, (recall, alarm) in sorted(ends.items(), key=lambda item: item[1][1]):
        placed = alarm if placed is None else max(alarm, placed + 0.035 * (high - low))
        axis.annotate("DR4 (variant)" if rule == DR4 else rule, (recall, alarm), xytext=(recall + 0.004, placed),
                      va="center", color=INK, fontsize=9,
                      arrowprops={"arrowstyle": "-", "color": MUTED, "linewidth": 0.8} if placed != alarm else None)
    axis.axvline(WHO_RECALL_TARGET, color=MUTED, linewidth=1.0, linestyle=(0, (4, 3)), zorder=1)
    axis.annotate("0.95 target", (WHO_RECALL_TARGET, 1), xycoords=("data", "axes fraction"), xytext=(4, -12),
                  textcoords="offset points", color=MUTED, fontsize=9)
    axis.set_title(f"Validation false alarms against WHO recall, every rule fitted on calibration ({report['run']})",
                   color=INK, fontsize=11, loc="left")
    axis.set_xlabel("WHO recall: share of WHO-listed photos called dangerous", color=MUTED)
    axis.set_ylabel("False alarms: share of photos neither WHO-listed\nnor viper or elapid that are called dangerous",
                    color=MUTED)
    axis.grid(color=GRID, linewidth=0.8)
    axis.set_axisbelow(True)
    for side in ("top", "right"):
        axis.spines[side].set_visible(False)
    for side in ("left", "bottom"):
        axis.spines[side].set_color(GRID)
    axis.tick_params(colors=MUTED)
    axis.margins(x=0.12)
    handles = axis.get_legend_handles_labels()
    marker_keys = [axis.plot([], [], marker="o", color=MUTED, linestyle="", markerfacecolor=face, markersize=size)[0]
                   for face, size in ((MUTED, MARKER_SIZE), (SURFACE, MARKER_SIZE), (MUTED, HEADLINE_MARKER_SIZE))]
    axis.legend(handles[0] + marker_keys, handles[1] + ["point fit", "lower-bound fit", "fitted at 0.95"],
                frameon=False, fontsize=8, loc="upper left", labelcolor=INK)
    figure.tight_layout()
    path.parent.mkdir(parents=True, exist_ok=True)
    figure.savefig(path, dpi=200, facecolor=SURFACE)

In [7]:
if DR_RUN:
    source = next(folder for handle in RUN_KERNELS for folder in run_folders(kernel_output(handle))
                  if folder.name == DR_RUN)
    predicted = load_config(source / "config.yaml").mask_source == PREDICTED
    dr = danger_rules_report(source, classes, OUT / "dr" / DR_RUN, OUT / "dr" / FIGURE_FILE.format(DR_RUN),
                             box_conf=box_conf if predicted else None)
    for rule, by_bound in dr["headline"]["rows"].items():
        print(f"{rule}: " + "; ".join(f"{bound} unreachable" if "unreachable" in row else
                                      f"{bound} recall {row['who_recall']:.3f} false alarms {row['false_alarm']:.3f} "
                                      f"o3 {row['o3']:.3f}" for bound, row in by_bound.items()))
    print(dr["verdict"])

DR0: point recall 0.931 false alarms 0.270 o3 0.028; lower recall 0.948 false alarms 0.311 o3 0.021
DR1: point recall 0.934 false alarms 0.268 o3 0.024; lower recall 0.954 false alarms 0.326 o3 0.016
DR2_top3: point recall 0.929 false alarms 0.327 o3 0.021; lower recall 0.939 false alarms 0.394 o3 0.015
DR2_top5: point recall 0.936 false alarms 0.288 o3 0.021; lower recall 0.950 false alarms 0.397 o3 0.013
DR3_max: point recall 0.924 false alarms 0.249 o3 0.028; lower recall 0.950 false alarms 0.314 o3 0.019
DR3_mix: point recall 0.933 false alarms 0.258 o3 0.024; lower recall 0.948 false alarms 0.302 o3 0.020
DR4: point recall 0.842 false alarms 0.162 o3 0.028; lower recall 0.842 false alarms 0.162 o3 0.021
no rule beats DR0
